# RVC 训练队列（Blue Archive 日配）

## 首次准备（只做一次）

1. 在 Google Drive 新建文件夹 `RVC-Train/`
2. 把本地 `AI-Models/datasets/` 里全部 zip 上传到 `RVC-Train/datasets/`

## 每次使用

1. 右上角 修改运行时类型 → T4 GPU
2. 运行下面"训练队列"框（第一次会弹 Google Drive 授权，点允许）
3. 之后不用管，断线/额度耗尽后重新打开本笔记再跑一次即可，自动续练

- 想先练某几个模型：在 PRIORITY 里填名字（逗号分隔，模糊匹配，如 `Hina,Alice`）
- 随时可运行最下面的"查看进度"框看训练状态
- 成品在 Drive 的 `RVC-Train/exported/<模型名>/` 下

In [ ]:
#@title 训练队列 { display-mode: "form" }
DRIVE_ROOT = "/content/drive/MyDrive/RVC-Train"  #@param {type:"string"}
REPO = "https://github.com/Kara251/rvc-colab.git"  #@param {type:"string"}
PRIORITY = ""  #@param {type:"string"}

import os
from google.colab import drive

drive.mount("/content/drive")

if not os.path.isdir("/content/rvc-colab"):
    !git clone --depth 1 {REPO} /content/rvc-colab
else:
    !git -C /content/rvc-colab pull

assert os.path.exists(os.path.join(DRIVE_ROOT, "datasets")), \
    "找不到 RVC-Train/datasets/，请先把数据集 zip 上传到 Drive"

os.environ["RVC_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["RVC_PRIORITY"] = PRIORITY
os.chdir("/content/rvc-colab")
!python worker.py

## 查看进度

随时可以运行下面这个框（不影响正在进行的训练）。

In [ ]:
#@title 查看进度 { display-mode: "form" }
import os, re, json, glob, time
import urllib.request, yaml

DRIVE_ROOT = globals().get("DRIVE_ROOT", "/content/drive/MyDrive/RVC-Train")
REPO = globals().get("REPO", "https://github.com/Kara251/rvc-colab.git")

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

local_manifest = "/content/rvc-colab/models.yaml"
if os.path.exists(local_manifest):
    manifest = yaml.safe_load(open(local_manifest, encoding="utf-8"))["models"]
else:
    raw = REPO.replace("github.com", "raw.githubusercontent.com").replace(".git", "") + "/main/models.yaml"
    manifest = yaml.safe_load(urllib.request.urlopen(raw))["models"]

def epoch_of(logdir):
    n = 0
    for f in glob.glob(os.path.join(logdir, "*_e_*.pth")) + glob.glob(os.path.join(logdir, "G_*.pth")):
        m = re.search(r"_(\d+)e_", os.path.basename(f))
        if m:
            n = max(n, int(m.group(1)))
    return n

done, doing, todo = [], [], []
for m in manifest:
    logdir = os.path.join(DRIVE_ROOT, "logs", m["name"])
    target = m.get("epochs", 0)
    if os.path.exists(os.path.join(logdir, ".trained")):
        done.append(m["name"])
    elif os.path.isdir(logdir):
        doing.append((m["name"], epoch_of(logdir), target))
    else:
        todo.append(m["name"])

st_path = os.path.join(DRIVE_ROOT, "status.json")
if os.path.exists(st_path):
    st = json.load(open(st_path))
    age = int(time.time() - st.get("ts", 0))
    print(f"当前: {st['model']} | 阶段 {st['phase']} | epoch {st['epoch']} | 心跳 {age} 秒前")
    if age > 300:
        print("  (心跳超过 5 分钟没更新，会话可能已断)")
    print()

print(f"已完成 {len(done)} / {len(manifest)}，训练中/有进度 {len(doing)}，未开始 {len(todo)}")
for name, ep, target in doing:
    print(f"  训练中 {name}: epoch {ep}/{target}")
if done:
    print("\n已完成:", ", ".join(done))